# Directional Misalignment Analysis

**The paradox:** Python-trained probes transfer to Java/C++/C# with 97–98% accuracy, yet the cosine similarity between Python and target-language probe weight vectors is only 0.15–0.30.

**Central question:** How can two probes point in very different directions in 1536-d space and yet both classify correctly?

## Hypotheses

| # | Hypothesis | Test |
|---|-----------|------|
| H1 | **Wide feasibility set** — data is so separable that many hyperplane orientations all work. `w_py` and `w_java` are two valid solutions from a large family. | Probe margin analysis |
| H2 | **Rotation in shared subspace** — both languages encode the role in a low-dimensional subspace; the subspaces are aligned even if the 1D probe vectors are not. | Principal subspace angles |
| H3 | **Class mean alignment** — index-token clusters have similar geometry across languages; the probe vectors differ but both point *toward* the same region. | Class mean direction cosines |
| H4 | **Different features, same label** — Python uses feature dimension A, Java uses dimension B, both predictive for different structural reasons. | Probe weight correlation + top-dimension overlap |

Results saved to `results/directional_analysis/`.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
# PCA for visualising the embedding space
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
# principal angle computation
from scipy.linalg import subspace_angles  # scipy gives the principal angles between two subspaces
from collections import defaultdict
import re, ast, os, json, warnings
warnings.filterwarnings('ignore')  # suppress the sklearn/tqdm noise — gets annoying fast

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'  # use GPU if it's there, otherwise fall back to CPU
# where all the plots and CSVs will land
RESULTS_DIR = '../results/directional_analysis'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Device: {DEVICE}')
print(f'Results → {RESULTS_DIR}')

Device: cpu
Results → ../results/directional_analysis


## Model + Data Loading

In [2]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
print(f'Loading {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)  # needed for Qwen — it has custom model code
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # Qwen has no separate pad token, eos works fine here
model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True, trust_remote_code=True)  # needed for Qwen — it has custom model code
model.eval().to(DEVICE)  # inference only, no gradients needed

# grab these once — reuse later
NUM_LAYERS      = model.config.num_hidden_layers
HIDDEN_SIZE     = model.config.hidden_size
_ids            = tokenizer('x', return_tensors='pt')['input_ids'][0].tolist()
LEADING_SPECIAL = len(_ids) - len(tokenizer.tokenize('x'))  # BERT has [CLS] so offset=1; Qwen has none so offset=0
print(f'Layers={NUM_LAYERS}  Hidden={HIDDEN_SIZE}  LeadingSpecial={LEADING_SPECIAL}')

Loading Qwen/Qwen2.5-1.5B ...


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 6829.27it/s]

Layers=28  Hidden=1536  LeadingSpecial=0


In [3]:
# adjust this path if the data is somewhere else
XLCOST_ROOT  = '/Applications/Projects/Algoverse/Coding/XLCoST_data'
NL2CODE_PROG = os.path.join(XLCOST_ROOT, 'retrieval', 'nl2code_search', 'program_level')

def reconstruct_code(tokens):  # XLCoST stores code as token lists with indent markers
    indent, lines, cur = 0, [], []
    for t in tokens:
        if t == 'NEW_LINE':  # XLCoST encodes newlines as explicit tokens
            lines.append('    ' * indent + ' '.join(cur)); cur = []
        elif t == 'INDENT':  indent += 1  # indentation is tracked via these special tokens
        elif t == 'DEDENT':  indent = max(0, indent - 1)
        else: cur.append(t)
    if cur: lines.append('    ' * indent + ' '.join(cur))
    return '\n'.join(lines).strip()

def load_programs(lang, split='train', n=300):
    path = os.path.join(NL2CODE_PROG, lang, f'{split}.jsonl')
    progs = []
    with open(path) as f:
        for line in f:
            if len(progs) >= n: break
            rec  = json.loads(line.strip())  # each line in the .jsonl is one program
            code = reconstruct_code(rec['code_tokens'])  # XLCoST stores code as token lists with indent markers
            if code: progs.append(code)
    return progs

# Index/key extractors per language
def get_py_idx(code):
    idx = set()
    try:
        for node in ast.walk(ast.parse(code)):  # AST is more reliable than regex for finding subscripts
            if isinstance(node, ast.Subscript):  # subscript nodes cover both arr[i] and d[key]
                slc = node.slice
                if isinstance(slc, ast.Index): slc = slc.value  # Python <3.9 wraps slice in an Index node, 3.9+ doesn't
                if isinstance(slc, ast.Name):  idx.add(slc.id)  # only care about variable names, not literals like arr[0]
    except SyntaxError: pass
    return idx

_BRACKET = re.compile(r'\[\s*([a-zA-Z_]\w*)\s*\]')
_JAVA_KW = {'int','long','char','byte','short','boolean','String','Integer',
             'Long','Object','null','true','false'}
_C_KW    = {'int','long','char','short','float','double','unsigned','void',
             'size_t','bool','true','false','null','NULL'}
_CS_KW   = {'int','long','char','byte','short','bool','float','double',
             'string','object','null','true','false','var'}
_JS_KW   = {'undefined','null','true','false','NaN','Infinity','length','prototype'}

def _bracket_idx(code, extra=None, exclude=None):
    names = {m.group(1) for m in _BRACKET.finditer(code)}
    if extra:
        for p in extra: names |= {m.group(1) for m in re.finditer(p, code)}
    return names - (exclude or set())

EXTRACTORS = {
    'Python':     get_py_idx,
    'Java':       lambda c: _bracket_idx(c, [r'\.(?:get|containsKey|getOrDefault)\(\s*([a-zA-Z_]\w*)'], _JAVA_KW),
    'C++':        lambda c: _bracket_idx(c, [r'\.(?:at|find|count)\(\s*([a-zA-Z_]\w*)'], _C_KW),
    'C':          lambda c: _bracket_idx(c, exclude=_C_KW),
    'C#':         lambda c: _bracket_idx(c, [r'\.(?:TryGetValue|ContainsKey|GetValueOrDefault)\(\s*([a-zA-Z_]\w*)'], _CS_KW),
    'Javascript': lambda c: _bracket_idx(c, [r'\.(?:get|has|delete)\(\s*([a-zA-Z_]\w*)'], _JS_KW),
}

def label_tokens(code, target_names):
    if not target_names: return [], []  # nothing to label if no index vars were found
    tokens, labels = tokenizer.tokenize(code), []
    for t in tokens:
        clean = t.lstrip('Ġ▁Ā').lstrip('##')  # GPT-style tokenizers prefix spaces with Ġ
        labels.append(1 if (not t.startswith('##')) and  # BERT wordpiece continuation tokens — skip these
                      (clean in target_names or clean.strip('[]().,;:') in target_names) else 0)
    return tokens, labels

print('Data utilities ready.')

Data utilities ready.


In [4]:
# progress bars — helpful when extraction takes a while
from tqdm.auto import tqdm

def build_and_extract(lang, n_programs=300):
    """Load programs, label tokens, extract hidden states for one language."""
    programs  = load_programs(lang, n=n_programs)
    extractor = EXTRACTORS[lang]
    dataset   = []
    for code in programs:
        names = extractor(code)
        if not names: continue
        toks, labs = label_tokens(code, names)
        if toks and sum(labs) > 0:
            dataset.append({'code': code, 'tokens': toks, 'labels': labs, 'target_names': names})

    all_h, all_l = defaultdict(list), []
    with torch.no_grad():  # saves memory — we're not doing any backprop
        for s in tqdm(dataset, desc=f'  {lang}', leave=False):
            enc  = tokenizer(s['code'], return_tensors='pt', truncation=True,
                             max_length=512, padding=False).to(DEVICE)  # no padding since we're processing one at a time
            n    = enc['input_ids'].shape[1] - LEADING_SPECIAL
            outs = model(**enc)
            start, end = LEADING_SPECIAL, LEADING_SPECIAL + n
            for li, lhs in enumerate(outs.hidden_states):
                all_h[li].extend(lhs[0, start:end, :].float().cpu().numpy())  # Qwen uses bfloat16 by default — numpy can't handle it, cast first
            all_l.extend(s['labels'][:n])

    for li in all_h:
        all_h[li] = np.stack(all_h[li])  # convert list of vectors to a proper 2D array
    labels = np.array(all_l)
    n_pos  = labels.sum()
    print(f'  {lang:12s}: {len(dataset)} programs | {len(labels):,} tokens | {n_pos} positive ({100*n_pos/len(labels):.1f}%)')
    return all_h, labels


def train_probe(hidden, labels, layer, random_state=42):
    """Train a single logistic regression probe at one layer."""
    X = hidden[layer]
    tr, te = train_test_split(np.arange(len(labels)), test_size=0.2,  # 80/20 split
                               random_state=random_state, stratify=labels)  # make sure both train and test have the same class ratio
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0, random_state=random_state)  # index tokens are ~6% of all tokens — need to reweight or probe ignores them
    clf.fit(X[tr], labels[tr])
    return clf, te


# ── Load all languages ────────────────────────────────────────────────────────
LANGUAGES = ['Python', 'Java', 'C++', 'C', 'C#', 'Javascript']
lang_hidden = {}
lang_labels = {}

print('Extracting hidden states for all languages...')
for lang in LANGUAGES:
    h, lbs = build_and_extract(lang, n_programs=300)
    lang_hidden[lang] = h
    lang_labels[lang] = lbs

print('\nDone.')

Extracting hidden states for all languages...


  Python:  14%|█▍        | 15/108 [01:32<08:04,  5.21s/it]

In [ ]:
# Train per-language probes at every layer + find best layer per language
lang_probes    = {}   # lang → {layer → clf}
lang_best_layer = {}

for lang in LANGUAGES:
    probes = {}
    best_f1, best_l = -1, -1
    for layer in tqdm(sorted(lang_hidden[lang].keys()), desc=f'  Probing {lang}', leave=False):
        clf, te = train_probe(lang_hidden[lang], lang_labels[lang], layer)
        y_pred  = clf.predict(lang_hidden[lang][layer][te])
        f1      = f1_score(lang_labels[lang][te], y_pred, average='macro')  # macro F1 is fairer given the class imbalance
        probes[layer] = clf
        if f1 > best_f1:
            best_f1, best_l = f1, layer
    lang_probes[lang]     = probes
    lang_best_layer[lang] = best_l
    print(f'  {lang:12s}: best layer={best_l}  F1={best_f1:.3f}')

---
## H1 — Wide Feasibility Set: Probe Margin Analysis

If the data is **very easily separable**, many hyperplane orientations all work — including both
`w_py` and `w_java`. The functional margin (`min |w·x + b| / ||w||`) measures how far the
closest point is from the decision boundary. A large margin means the separator has room to
rotate without misclassifying anything.

In [ ]:
def compute_margins(clf, X, y):
    """Functional margin statistics for each class."""
    w, b    = clf.coef_[0], clf.intercept_[0]
    scores  = X @ w + b                        # signed distance × ||w||
    norm_w  = np.linalg.norm(w)
    margins = scores / norm_w                  # functional margin
    pos_margins = margins[y == 1]
    neg_margins = margins[y == 0]
    return {
        'min_pos': pos_margins.min(),
        'min_neg': -neg_margins.max(),         # closest negative to boundary
        'mean_pos': pos_margins.mean(),
        'mean_neg': -neg_margins.mean(),
        'geometric_margin': min(pos_margins.min(), -neg_margins.max()),
    }


# Also compute cross-language transfer F1 and cosine at best Python layer
py_layer = lang_best_layer['Python']
py_probe = lang_probes['Python'][py_layer]
py_w     = py_probe.coef_[0]

rows = []
for lang in LANGUAGES:
    layer  = lang_best_layer['Python']  # use Python's best layer uniformly
    X_lang = lang_hidden[lang][layer]
    y_lang = lang_labels[lang]

    # Cross-transfer: apply Python probe to this language
    xfer_f1  = f1_score(y_lang, py_probe.predict(X_lang), average='macro')  # macro F1 is fairer given the class imbalance
    xfer_acc = accuracy_score(y_lang, py_probe.predict(X_lang))

    # Margin of Python probe on this language's data
    marg = compute_margins(py_probe, X_lang, y_lang)

    # Cosine between Python probe and this language's own probe
    own_probe = lang_probes[lang][layer]
    own_w     = own_probe.coef_[0]
    cosine    = np.dot(py_w, own_w) / (np.linalg.norm(py_w) * np.linalg.norm(own_w))

    rows.append({
        'lang':            lang,
        'xfer_f1':         xfer_f1,
        'xfer_acc':        xfer_acc,
        'cosine_w':        cosine,
        'geo_margin':      marg['geometric_margin'],  # how much room the boundary has to rotate before misclassifying
        'mean_pos_margin': marg['mean_pos'],
        'mean_neg_margin': marg['mean_neg'],
    })

print(f'{"Lang":12s}  {"Xfer F1":>9}  {"Xfer Acc":>9}  {"Cosine(w)": >11}  {"Geo Margin":>11}  {"Mean+Margin":>12}')
print('-' * 75)
for r in rows:
    print(f'{r["lang"]:12s}  {r["xfer_f1"]:>9.3f}  {r["xfer_acc"]:>9.3f}  '
          f'{r["cosine_w"]:>11.4f}  {r["geo_margin"]:>11.4f}  {r["mean_pos_margin"]:>12.4f}')  # how much room the boundary has to rotate before misclassifying

In [ ]:
# ── Scatter: cosine(w) vs geometric margin ────────────────────────────────────
LANG_COLORS = {'Python':'#FF5722','Java':'#FF9800','C++':'#2196F3',
               'C':'#4CAF50','C#':'#9C27B0','Javascript':'#F44336'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: cosine vs geometric margin
ax = axes[0]
for r in rows:
    ax.scatter(r['cosine_w'], r['geo_margin'],  # how much room the boundary has to rotate before misclassifying
               color=LANG_COLORS[r['lang']], s=120, zorder=3,
               label=r['lang'])
    ax.annotate(r['lang'], (r['cosine_w'], r['geo_margin']),  # how much room the boundary has to rotate before misclassifying
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel('Cosine Similarity (Python vs Lang probe direction)', fontsize=11)
ax.set_ylabel('Geometric Margin (Python probe on lang data)', fontsize=11)
ax.set_title('H1: Is misalignment explained by wide margins?', fontsize=12)
ax.grid(True, alpha=0.3)  # slightly transparent so overlapping points are visible

# Right: cosine vs cross-transfer F1
ax = axes[1]
for r in rows:
    ax.scatter(r['cosine_w'], r['xfer_f1'],
               color=LANG_COLORS[r['lang']], s=120, zorder=3)
    ax.annotate(r['lang'], (r['cosine_w'], r['xfer_f1']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel('Cosine Similarity (probe directions)', fontsize=11)
ax.set_ylabel('Cross-Language Transfer F1', fontsize=11)
ax.set_title('Low cosine ≠ poor transfer', fontsize=12)
ax.grid(True, alpha=0.3)  # slightly transparent so overlapping points are visible

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/h1_margin_vs_cosine.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: h1_margin_vs_cosine.png')

---
## H2 — Subspace Alignment: Principal Angles

Instead of comparing 1-D probe vectors, compare the **k-dimensional subspaces** spanned by the
top-k principal components of index-token embeddings in each language.

**Principal angle** between subspaces A and B: `θ_i = arccos(σ_i(A^T B))`  
θ close to 0 = aligned; θ close to 90° = orthogonal.

If H2 is true, principal angles should be small (subspaces aligned) even though the 1D probe
vectors disagree.

In [ ]:
def top_k_subspace(X, k=10):  # take the top-k right singular vectors as the subspace basis
    """Return the top-k right singular vectors (principal directions) of X."""
    X_centered = X - X.mean(axis=0)
    _, _, Vt   = np.linalg.svd(X_centered, full_matrices=False)
    return Vt[:k].T   # shape (hidden_size, k)


K_VALUES = [1, 5, 10, 20, 50]
layer    = lang_best_layer['Python']

# Build subspaces from index-only token embeddings
lang_index_subspace = {}
for lang in LANGUAGES:
    X   = lang_hidden[lang][layer]
    y   = lang_labels[lang]
    X_pos = X[y == 1]   # index/key tokens only
    lang_index_subspace[lang] = X_pos
    print(f'  {lang:12s}: {X_pos.shape[0]} index tokens')

# Compute principal angles: Python vs each other language
py_X_pos = lang_index_subspace['Python']

print(f'\nPrincipal angles (degrees) — Python vs other languages')
print(f'  k-dim subspace of INDEX-TOKEN embeddings at layer {layer}')
print()

angle_table = {}
for lang in LANGUAGES:
    if lang == 'Python': continue
    tgt_X_pos = lang_index_subspace[lang]
    angles_per_k = []
    for k in K_VALUES:
        k_actual = min(k, py_X_pos.shape[0]-1, tgt_X_pos.shape[0]-1, HIDDEN_SIZE)
        S_py  = top_k_subspace(py_X_pos,  k_actual)  # take the top-k right singular vectors as the subspace basis
        S_tgt = top_k_subspace(tgt_X_pos, k_actual)  # take the top-k right singular vectors as the subspace basis
        angles_rad = subspace_angles(S_py, S_tgt)          # scipy: principal angles
        mean_angle = np.degrees(angles_rad.mean())
        angles_per_k.append(mean_angle)
    angle_table[lang] = angles_per_k

header = f'{"Language":12s}  ' + '  '.join(f'k={k:2d}' for k in K_VALUES)
print(header)
print('-' * len(header))
for lang, angles in angle_table.items():
    row = f'{lang:12s}  ' + '  '.join(f'{a:5.1f}°' for a in angles)
    print(row)

In [ ]:
# ── Plot: mean principal angle vs k ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for lang, angles in angle_table.items():
    ax.plot(K_VALUES, angles, marker='o', color=LANG_COLORS[lang],
            linewidth=2, markersize=6, label=lang)
ax.axhline(90, color='gray', linestyle='--', alpha=0.4, label='Orthogonal (90°)')
ax.axhline(45, color='gray', linestyle=':',  alpha=0.4, label='Random (≈45°)')
ax.set_xlabel('Subspace dimension k', fontsize=11)
ax.set_ylabel('Mean principal angle (°)', fontsize=11)
ax.set_title('H2: Index-token subspace alignment\n(Python vs other languages)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)  # slightly transparent so overlapping points are visible

# Right: compare cosine(probe vectors) vs mean principal angle at k=10
ax = axes[1]
k_idx = K_VALUES.index(10)
for r in rows:
    if r['lang'] == 'Python': continue
    ang = angle_table[r['lang']][k_idx]
    ax.scatter(r['cosine_w'], ang,
               color=LANG_COLORS[r['lang']], s=120, zorder=3)
    ax.annotate(r['lang'], (r['cosine_w'], ang),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel('Cosine similarity (1-D probe vectors)', fontsize=11)
ax.set_ylabel('Mean principal angle k=10 (°)', fontsize=11)
ax.set_title('1-D probe cosine vs k-D subspace angle', fontsize=12)
ax.grid(True, alpha=0.3)  # slightly transparent so overlapping points are visible

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/h2_principal_angles.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: h2_principal_angles.png')

---
## H3 — Class Mean Direction

Instead of probe weight vectors, compare the **class mean direction**:  
`d_lang = mean(index tokens) - mean(non-index tokens)`

The probe weight vector is a learned direction; the class mean direction is directly computed
from the data. If class mean directions are more aligned across languages than probe directions,
it suggests probes overfit to language-specific noise while the underlying geometry is shared.

In [ ]:
layer = lang_best_layer['Python']

def class_mean_direction(X, y):  # mean(positive) - mean(negative), then normalize
    """Returns the unit vector from mean(class 0) to mean(class 1)."""
    d = X[y == 1].mean(axis=0) - X[y == 0].mean(axis=0)
    return d / np.linalg.norm(d)

lang_cmd   = {}   # class mean direction per language
lang_means = {}   # {'pos': ..., 'neg': ...}

for lang in LANGUAGES:
    X, y = lang_hidden[lang][layer], lang_labels[lang]
    lang_cmd[lang]   = class_mean_direction(X, y)  # mean(positive) - mean(negative), then normalize
    lang_means[lang] = {'pos': X[y==1].mean(0), 'neg': X[y==0].mean(0)}

py_cmd = lang_cmd['Python']
py_w   = lang_probes['Python'][layer].coef_[0]
py_w   = py_w / np.linalg.norm(py_w)

print(f'{"Language":12s}  {"Cosine(probe_w)": >17}  {"Cosine(class_mean)": >20}  '
      f'{"Probe‖w‖/CMD": >14}')
print('-' * 70)
for lang in LANGUAGES:
    cos_probe = np.dot(py_w, lang_probes[lang][layer].coef_[0] /
                       np.linalg.norm(lang_probes[lang][layer].coef_[0]))
    cos_cmd   = np.dot(py_cmd, lang_cmd[lang])
    print(f'{lang:12s}  {cos_probe:>17.4f}  {cos_cmd:>20.4f}')

In [ ]:
# ── Bar chart: probe cosine vs class-mean-direction cosine ───────────────────
langs_no_py = [l for l in LANGUAGES if l != 'Python']

probe_cosines = []
cmd_cosines   = []
for lang in langs_no_py:
    pw = lang_probes[lang][layer].coef_[0]
    probe_cosines.append(np.dot(py_w, pw / np.linalg.norm(pw)))
    cmd_cosines.append(np.dot(py_cmd, lang_cmd[lang]))

x = np.arange(len(langs_no_py))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, probe_cosines, w, label='Probe weight cosine', color='#FF5722', alpha=0.85)
b2 = ax.bar(x + w/2, cmd_cosines,   w, label='Class-mean direction cosine', color='#2196F3', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(langs_no_py, fontsize=11)
ax.set_ylabel('Cosine similarity with Python', fontsize=11)
ax.set_title('H3: Probe directions vs class-mean directions (Python baseline)\n'
             'If blue > orange: underlying geometry is more shared than probes suggest', fontsize=12)
ax.axhline(0, color='black', linewidth=0.8)  # zero line for reference
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)  # slightly transparent so overlapping points are visible
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/h3_class_mean_direction.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: h3_class_mean_direction.png')  # mean(positive) - mean(negative), then normalize

---
## H4 — Different Features, Same Label

Do Python and Java probes rely on the **same hidden dimensions**?  
Compare the top-N most influential dimensions in each probe weight vector.

High overlap → same features, aligned probes should be similar  
Low overlap → different features explain why cosine is low despite both working

In [ ]:
TOP_K_DIMS = [10, 50, 100, 200, 500]

def top_k_dims(w, k):
    """Return indices of top-k dimensions by absolute weight magnitude."""
    return set(np.argsort(np.abs(w))[-k:])

def jaccard(a, b):  # overlap of top-k most-weighted dimensions
    return len(a & b) / len(a | b)

def expected_jaccard(k, d):  # overlap of top-k most-weighted dimensions
    """Expected Jaccard of two random k-subsets from d dimensions."""
    return k / (2*d - k)

py_w_raw = lang_probes['Python'][layer].coef_[0]

print(f'Top-k dimension overlap (Jaccard) — Python probe vs other languages')
print(f'(Random baseline Jaccard shown in parentheses)')
print()
print(f'{"Language":12s}  ' + '  '.join(f'k={k:3d}' for k in TOP_K_DIMS))
print('-' * (12 + 11 * len(TOP_K_DIMS)))

jaccard_table = {}  # overlap of top-k most-weighted dimensions
for lang in LANGUAGES:
    if lang == 'Python': continue
    w_lang  = lang_probes[lang][layer].coef_[0]
    row_j   = []
    for k in TOP_K_DIMS:
        py_dims   = top_k_dims(py_w_raw, k)
        lang_dims = top_k_dims(w_lang, k)
        j         = jaccard(py_dims, lang_dims)  # overlap of top-k most-weighted dimensions
        row_j.append(j)
    jaccard_table[lang] = row_j  # overlap of top-k most-weighted dimensions
    rand_j = [f'({expected_jaccard(k, HIDDEN_SIZE):.3f})' for k in TOP_K_DIMS]  # overlap of top-k most-weighted dimensions
    print(f'{lang:12s}  ' + '  '.join(f'{j:.3f}' for j in row_j))
print('  (random)  ' + '  '.join(f'{expected_jaccard(k, HIDDEN_SIZE):.3f}' for k in TOP_K_DIMS))  # overlap of top-k most-weighted dimensions

In [ ]:
# ── Plot: Jaccard overlap vs k ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for lang, js in jaccard_table.items():  # overlap of top-k most-weighted dimensions
    ax.plot(TOP_K_DIMS, js, marker='o', color=LANG_COLORS[lang],
            linewidth=2, markersize=6, label=lang)

# Random baseline
rand_js = [expected_jaccard(k, HIDDEN_SIZE) for k in TOP_K_DIMS]  # overlap of top-k most-weighted dimensions
ax.plot(TOP_K_DIMS, rand_js, color='gray', linestyle='--',
        linewidth=1.5, label='Random baseline')

ax.set_xlabel('Top-k dimensions', fontsize=11)
ax.set_ylabel('Jaccard similarity', fontsize=11)
ax.set_title('H4: Do Python and target-language probes use the same hidden dimensions?\n'
             'Above random → shared feature dimensions', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)  # slightly transparent so overlapping points are visible
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/h4_dimension_overlap.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: h4_dimension_overlap.png')

---
## PCA Visualization — Shared Embedding Space

Fit PCA jointly on all languages' hidden states at the best Python layer.
Visualize index vs non-index clusters per language in the shared 2D space.

**If H2 is correct:** index-token clusters should overlap across languages,
even if they're rotated relative to the probe directions.

In [ ]:
layer     = lang_best_layer['Python']
N_SAMPLE  = 500   # sample per class per language to keep plot readable

# Pool all data to fit a shared PCA
all_X, all_y, all_langs = [], [], []
rng = np.random.default_rng(42)

for lang in LANGUAGES:
    X, y = lang_hidden[lang][layer], lang_labels[lang]
    for cls in [0, 1]:
        idx   = np.where(y == cls)[0]
        idx   = rng.choice(idx, size=min(N_SAMPLE, len(idx)), replace=False)
        all_X.append(X[idx])
        all_y.extend([cls] * len(idx))
        all_langs.extend([lang] * len(idx))

all_X     = np.vstack(all_X)
all_y     = np.array(all_y)
all_langs = np.array(all_langs)

print(f'Fitting PCA on {len(all_X):,} tokens from all languages...')
pca = PCA(n_components=2)
all_X2 = pca.fit_transform(all_X)  # fit on pooled data so all languages share the same axes
print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}  PC2={pca.explained_variance_ratio_[1]:.3f}')

In [ ]:
# ── Plot 1: colored by language, shaped by class ─────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# Compute per-language PCA projections
lang_offset = 0
lang_ranges = {}
for lang in LANGUAGES:
    n = (all_langs == lang).sum()
    lang_ranges[lang] = (lang_offset, lang_offset + n)
    lang_offset += n

for i, lang in enumerate(LANGUAGES):
    ax      = axes[i]
    mask    = all_langs == lang
    X2_lang = all_X2[mask]
    y_lang  = all_y[mask]

    ax.scatter(X2_lang[y_lang==0, 0], X2_lang[y_lang==0, 1],
               c='#BBDEFB', s=8, alpha=0.4, label='non-index')
    ax.scatter(X2_lang[y_lang==1, 0], X2_lang[y_lang==1, 1],
               c=LANG_COLORS[lang], s=20, alpha=0.8, label='index/key')

    # Mark class means
    ax.scatter(*X2_lang[y_lang==0].mean(0), c='gray',  marker='*', s=200, zorder=5)
    ax.scatter(*X2_lang[y_lang==1].mean(0), c=LANG_COLORS[lang], marker='*', s=200, zorder=5)

    ax.set_title(f'{lang}', fontsize=12)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle(f'PCA of hidden states (layer {layer}) — all languages\n'
             'Shared PCA fitted on pooled data.  ★ = class mean.', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/pca_per_language.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: pca_per_language.png')

In [ ]:
# ── Plot 2: overlay index tokens only — do clusters coincide across languages? ─
fig, ax = plt.subplots(figsize=(10, 8))

for lang in LANGUAGES:
    mask       = (all_langs == lang) & (all_y == 1)
    X2_idx     = all_X2[mask]
    ax.scatter(X2_idx[:, 0], X2_idx[:, 1],
               c=LANG_COLORS[lang], s=15, alpha=0.5, label=lang)
    # 95th percentile ellipse via covariance
    mu   = X2_idx.mean(0)
    cov  = np.cov(X2_idx.T)
    from matplotlib.patches import Ellipse
    vals, vecs = np.linalg.eigh(cov)  # eigh for symmetric matrices — more stable than eig
    order  = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle  = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * 2.0 * np.sqrt(vals)  # 2-sigma
    ell = Ellipse(mu, width, height, angle=angle,  # 2-sigma confidence ellipse for each cluster
                  edgecolor=LANG_COLORS[lang], facecolor='none', linewidth=2)
    ax.add_patch(ell)

ax.set_xlabel('PC1', fontsize=11)
ax.set_ylabel('PC2', fontsize=11)
ax.set_title('Index/key token clusters — all languages in shared PCA space\n'
             'Ellipse = 2σ contour.  Overlap → shared geometry.', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/pca_index_token_overlay.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: pca_index_token_overlay.png')

---
## Layer-Wise Alignment Trajectory

Track how all four metrics evolve across layers:
1. Probe-direction cosine (1-D)
2. Mean principal angle (k=10 subspace)
3. Class-mean-direction cosine
4. Cross-transfer F1

This reveals **when** in the network the geometry converges or diverges.

In [ ]:
TARGET_LANGS = [l for l in LANGUAGES if l != 'Python']
layers_all   = sorted(lang_hidden['Python'].keys())

# For each layer, compute all 4 metrics for each target language
metrics_by_layer = {lang: {'cosine_w':[], 'subspace_angle':[], 'cmd_cosine':[], 'xfer_f1':[]}
                    for lang in TARGET_LANGS}

for layer in tqdm(layers_all, desc='Layer-wise analysis'):
    py_X  = lang_hidden['Python'][layer]
    py_y  = lang_labels['Python']
    py_w  = lang_probes['Python'][layer].coef_[0]
    py_w  = py_w / np.linalg.norm(py_w)
    py_cmd = class_mean_direction(py_X, py_y)  # mean(positive) - mean(negative), then normalize

    for lang in TARGET_LANGS:
        tgt_X  = lang_hidden[lang][layer]
        tgt_y  = lang_labels[lang]
        tgt_w  = lang_probes[lang][layer].coef_[0]
        tgt_w  = tgt_w / np.linalg.norm(tgt_w)
        tgt_cmd = class_mean_direction(tgt_X, tgt_y)  # mean(positive) - mean(negative), then normalize

        # 1. Probe cosine
        metrics_by_layer[lang]['cosine_w'].append(np.dot(py_w, tgt_w))

        # 2. Subspace angle (k=10)
        k  = min(10, py_X[py_y==1].shape[0]-1, tgt_X[tgt_y==1].shape[0]-1)
        S_py  = top_k_subspace(py_X[py_y==1],   k)  # take the top-k right singular vectors as the subspace basis
        S_tgt = top_k_subspace(tgt_X[tgt_y==1], k)  # take the top-k right singular vectors as the subspace basis
        ang   = np.degrees(subspace_angles(S_py, S_tgt).mean())  # scipy gives the principal angles between two subspaces
        metrics_by_layer[lang]['subspace_angle'].append(ang)

        # 3. Class mean direction cosine
        metrics_by_layer[lang]['cmd_cosine'].append(np.dot(py_cmd, tgt_cmd))

        # 4. Cross-transfer F1
        y_pred = lang_probes['Python'][layer].predict(tgt_X)
        metrics_by_layer[lang]['xfer_f1'].append(f1_score(tgt_y, y_pred, average='macro'))  # macro F1 is fairer given the class imbalance

print('Layer-wise metrics computed.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
metric_info = [
    ('cosine_w',      'Probe direction cosine (1-D)',     axes[0,0], False),
    ('subspace_angle','Mean principal angle k=10 (°)',    axes[0,1], True),
    ('cmd_cosine',    'Class-mean direction cosine',      axes[1,0], False),
    ('xfer_f1',       'Cross-language transfer F1',       axes[1,1], False),
]

for key, ylabel, ax, invert in metric_info:
    for lang in TARGET_LANGS:
        vals = metrics_by_layer[lang][key]
        ax.plot(layers_all, vals, marker='o', markersize=3,
                color=LANG_COLORS[lang], linewidth=1.8, label=lang)
    ax.set_xlabel('Layer Index', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(ylabel, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)  # slightly transparent so overlapping points are visible
    if key == 'subspace_angle':
        ax.axhline(90, color='gray', linestyle='--', alpha=0.4, label='Orthogonal')
        ax.axhline(45, color='gray', linestyle=':',  alpha=0.3, label='Random')  # slightly transparent so overlapping points are visible

plt.suptitle('Layer-wise alignment trajectory — Python vs other languages\n'
             'When does geometry converge / diverge?', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/layerwise_alignment_trajectory.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: layerwise_alignment_trajectory.png')

---
## Summary & Interpretation

In [ ]:
best_layer = lang_best_layer['Python']

print('=' * 72)
print('DIRECTIONAL MISALIGNMENT SUMMARY')
print(f'Layer analysed: {best_layer}  |  Model: {MODEL_NAME}')
print('=' * 72)

print(f'\n{"Language":12s}  {"Xfer F1":>8}  {"Probe cos":>10}  {"CMD cos":>9}  '
      f'{"SubspAng(10)":>14}  {"DimJacc(50)":>12}')
print('-' * 72)

for lang in TARGET_LANGS:
    r    = next(x for x in rows if x['lang'] == lang)
    cmd  = np.dot(lang_cmd['Python'], lang_cmd[lang])
    sang = angle_table[lang][K_VALUES.index(10)]
    jac  = jaccard_table[lang][TOP_K_DIMS.index(50)]  # overlap of top-k most-weighted dimensions
    print(f'{lang:12s}  {r["xfer_f1"]:>8.3f}  {r["cosine_w"]:>10.4f}  {cmd:>9.4f}  '
          f'{sang:>14.1f}°  {jac:>12.3f}')

print()
print('Interpretation:')
print('  Probe cos   — cosine between Python and target probe weight vectors (1-D)')
print('  CMD cos     — cosine between class-mean directions (more geometric)')
print('  SubspAng    — mean principal angle of top-10 subspaces (0°=aligned, 90°=orthogonal)')
print('  DimJacc(50) — Jaccard of top-50 most-weighted dimensions (random≈0.017)')
print()
print('Key finding:')
print('  If CMD cos >> Probe cos  → underlying geometry IS shared; probes pick up noise')
print('  If SubspAng << 45°       → index-token subspaces are aligned despite misaligned probes')
print('  If DimJacc >> random     → same hidden dimensions drive the decision in both languages')

In [ ]:
import csv

# Save layer-wise metrics to CSV
csv_path = f'{RESULTS_DIR}/layerwise_metrics.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['language','layer','cosine_w','subspace_angle_k10',
                     'cmd_cosine','xfer_f1'])
    for lang in TARGET_LANGS:
        m = metrics_by_layer[lang]
        for i, layer in enumerate(layers_all):
            writer.writerow([lang, layer,
                             f'{m["cosine_w"][i]:.4f}',
                             f'{m["subspace_angle"][i]:.2f}',
                             f'{m["cmd_cosine"][i]:.4f}',
                             f'{m["xfer_f1"][i]:.4f}'])
print(f'Saved: {csv_path}')

# Save summary row per language
summ_path = f'{RESULTS_DIR}/summary_per_language.csv'
with open(summ_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['language','xfer_f1','probe_cosine','cmd_cosine',
                     'subspace_angle_k10','jaccard_top50','geo_margin'])  # overlap of top-k most-weighted dimensions
    for lang in TARGET_LANGS:
        r    = next(x for x in rows if x['lang'] == lang)
        cmd  = np.dot(lang_cmd['Python'], lang_cmd[lang])
        sang = angle_table[lang][K_VALUES.index(10)]
        jac  = jaccard_table[lang][TOP_K_DIMS.index(50)]  # overlap of top-k most-weighted dimensions
        writer.writerow([lang, f'{r["xfer_f1"]:.4f}', f'{r["cosine_w"]:.4f}',
                         f'{cmd:.4f}', f'{sang:.2f}', f'{jac:.4f}',
                         f'{r["geo_margin"]:.4f}'])  # how much room the boundary has to rotate before misclassifying
print(f'Saved: {summ_path}')
print(f'\nAll outputs in: {RESULTS_DIR}')